In [1]:
%cd ProyectoFinal

/home/jovyan/work/ProyectoFinal


In [2]:
import pandas as pd
import requests
import json
from datetime import datetime
import os
import re
import subprocess
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import shutil
import holidays
from datetime import date

In [31]:
fin = date.today()
inicio = (pd.to_datetime(fin) - pd.DateOffset(months=6, day=1)).date()

print(fin)
print(inicio)

2026-04-27
2025-10-01


# Datos Aemet

In [ ]:
!conda install -c conda-forge unrar -y

In [39]:

# --- CONSTANTES ---
CONFIG = {
    "URL_WEB": "https://datosclima.es/Aemet2013/DescargaDatos.html",
    "BASE_URL": "https://datosclima.es/Aemet2013/",
    "TEMP_DIR": os.path.abspath("data/datos_clima_aemet"),
    "ARCHIVO_FINAL": os.path.abspath("data/datos_clima_aemet/datos_clima_2014_2026.csv"),
    "ULTIMA_FECHA_REGISTRADA": os.path.abspath("data/datos_clima_aemet/ultima_fecha_registrada.csv"),
    "HEADERS": {'User-Agent': 'Mozilla/5.0'}
}

def inicializar_entorno():
    """Crea directorios necesarios si no existen."""
    os.makedirs(CONFIG["TEMP_DIR"], exist_ok=True)

def obtener_ultima_fecha_registrada():
    """Detecta la última fecha en el CSV sin cargar todo el archivo en memoria."""
    if not os.path.exists(CONFIG["ARCHIVO_FINAL"]):
        return 0
    if not os.path.exists(CONFIG["ULTIMA_FECHA_REGISTRADA"]):
        try:
            # Leemos solo la última fila para optimizar memoria
            df_last = pd.read_csv(CONFIG["ARCHIVO_FINAL"], skipinitialspace=True).tail(1)
            if df_last.empty or 'fecha' not in df_last.columns:
                return 0
                
            fecha_dt = pd.to_datetime(df_last['fecha'].iloc[0], errors='coerce')
            if pd.notnull(fecha_dt):
                print(f"📅 Último registro local: {fecha_dt.date()}")
                return int(fecha_dt.strftime('%Y%m'))
        except Exception as e:
            print(f"⚠️ Error leyendo histórico: {e}")
        return 0
    
    try:
        # Leemos solo la última fila para optimizar memoria
        df_last = pd.read_csv(CONFIG["ULTIMA_FECHA_REGISTRADA"], skipinitialspace=True).tail(1)
        if df_last.empty or 'fecha' not in df_last.columns:
            return 0
            
        fecha_dt = pd.to_datetime(df_last['fecha'].iloc[0], errors='coerce')
        if pd.notnull(fecha_dt):
            print(f"📅 Último registro local: {fecha_dt.date()}")
            return int(fecha_dt.strftime('%Y%m'))
    except Exception as e:
        print(f"⚠️ Error leyendo histórico: {e}")
    return 0

def listar_archivos_pendientes(session, ultima_fecha):
    """Escanea la web y devuelve URLs de archivos posteriores a la última fecha."""
    res = session.get(CONFIG["URL_WEB"])
    soup = BeautifulSoup(res.text, 'html.parser')
    
    pendientes = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if '.rar' in href.lower():
            url = urljoin(CONFIG["BASE_URL"], href)
            nombre = url.split('/')[-1]
            
            # Regex para extraer AemetYYYY-MM
            match = re.search(r'20(\d{2})-(\d{2})', nombre)
            if match:
                fecha_int = int(f"20{match.group(1)}{match.group(2)}")
                if fecha_int >= 201401 and fecha_int > ultima_fecha:
                    pendientes.append((url, fecha_int))
    
    return sorted(pendientes, key=lambda x: x[1])

def procesar_excel(ruta_excel):
    """Limpia y extrae datos de un archivo Excel individual."""
    try:
        engine = 'xlrd' if ruta_excel.endswith('.xls') else 'openpyxl'
        df = pd.read_excel(ruta_excel, engine=engine, header=4)
        
        # Limpieza de columnas y filas
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')].dropna(how='all')
        
        # Lógica de fecha multi-formato
        nombre_archivo = os.path.basename(ruta_excel)
        digitos = "".join(filter(str.isdigit, nombre_archivo))
        
        if len(digitos) == 8:
            # Intentar YYYYMMDD (Nuevo) luego DDMMYYYY (Viejo)
            fecha_obj = pd.to_datetime(digitos, format='%Y%m%d', errors='coerce')
            if pd.isnull(fecha_obj) or fecha_obj.year < 2014:
                fecha_obj = pd.to_datetime(digitos, format='%d%m%Y', errors='coerce')
            
            if pd.notnull(fecha_obj):
                df['fecha'] = fecha_obj.strftime('%Y-%m-%d')
                return df
    except Exception as e:
        print(f"  ⚠️ Error en {os.path.basename(ruta_excel)}: {e}")
    return None

def ejecutar_actualizacion():
    inicializar_entorno()
    session = requests.Session()
    session.headers.update(CONFIG["HEADERS"])
    
    ultima_fecha = obtener_ultima_fecha_registrada()
    pendientes = listar_archivos_pendientes(session, ultima_fecha)
    
    if not pendientes:
        print("✅ Sistema al día. No se requiere descarga.")
        return

    print(f"🚀 Iniciando actualización incremental: {len(pendientes)} paquetes nuevos.")
    
    df_acumulado = []
    
    for url, _ in pendientes:
        nombre_rar = url.split('/')[-1]
        ruta_rar = os.path.join(CONFIG["TEMP_DIR"], nombre_rar)
        ext_dir = os.path.join(CONFIG["TEMP_DIR"], "temp_work")
        
        try:
            print(f"⬇️ Descargando: {nombre_rar}")
            resp = session.get(url)
            with open(ruta_rar, 'wb') as f:
                f.write(resp.content)
            
            os.makedirs(ext_dir, exist_ok=True)
            subprocess.run(['unrar', 'x', '-o+', ruta_rar, ext_dir], capture_output=True, check=True)
            
            for root, _, files in os.walk(ext_dir):
                for f in files:
                    if f.lower().endswith(('.xls', '.xlsx')):
                        df_res = procesar_excel(os.path.join(root, f))
                        if df_res is not None:
                            df_acumulado.append(df_res)
                            
        except Exception as e:
            print(f"💥 Fallo crítico procesando {nombre_rar}: {e}")
        finally:
            # Limpieza garantizada de archivos temporales
            if os.path.exists(ext_dir): shutil.rmtree(ext_dir)
            if os.path.exists(ruta_rar): os.remove(ruta_rar)
    
    df_clima = pd.DataFrame()

    if df_acumulado:
        print("💾 Consolidando datos...")
        # Cargar histórico si existe para unirlo
        df_clima = pd.concat(df_acumulado, ignore_index=True)
        
        if os.path.exists(CONFIG["ARCHIVO_FINAL"]):
            df_hist = pd.read_csv(CONFIG["ARCHIVO_FINAL"], low_memory=False)
            df_clima = pd.concat([df_hist, df_clima], ignore_index=True)
        
        # Eliminación de duplicados por seguridad (Estación + Fecha)
        df_clima.drop_duplicates(subset=['fecha', 'Estación'], keep='last', inplace=True)
        
        # Guardado eficiente
        df_clima.to_csv(CONFIG["ARCHIVO_FINAL"], index=False, encoding='utf-8-sig')
        df_clima["fecha"].tail(1).to_csv(CONFIG["ULTIMA_FECHA_REGISTRADA"], index=False, encoding='utf-8-sig')
        print(f"✅ Proceso completado. Archivo actualizado: {len(df_clima)} registros totales.")
    else:
        print("ℹ️ No se extrajeron nuevos datos válidos.")


ejecutar_actualizacion()

📅 Último registro local: 2026-03-31
✅ Sistema al día. No se requiere descarga.


In [ ]:
import os
import re
import shutil
import subprocess
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- CONSTANTES SIMPLIFICADAS ---
CONFIG = {
    "URL_WEB": "https://datosclima.es/Aemet2013/DescargaDatos.html",
    "BASE_URL": "https://datosclima.es/Aemet2013/",
    "TEMP_DIR": os.path.abspath("data/temp_descarga"),
    "ARCHIVO_SALIDA": os.path.abspath("data/datos_ultimos_5_meses.csv"),
    "HEADERS": {'User-Agent': 'Mozilla/5.0'}
}

def inicializar_entorno():
    """Crea y limpia el directorio temporal."""
    if os.path.exists(CONFIG["TEMP_DIR"]):
        shutil.rmtree(CONFIG["TEMP_DIR"])
    os.makedirs(CONFIG["TEMP_DIR"], exist_ok=True)

def listar_ultimos_5_rars(session):
    """Busca todos los .rar y devuelve las URLs de los 5 más recientes."""
    res = session.get(CONFIG["URL_WEB"])
    soup = BeautifulSoup(res.text, 'html.parser')
    
    enlaces = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if '.rar' in href.lower():
            url = urljoin(CONFIG["BASE_URL"], href)
            # Extraer fecha para ordenar correctamente (formato YYYY-MM)
            match = re.search(r'20(\d{2})-(\d{2})', url)
            if match:
                fecha_int = int(f"20{match.group(1)}{match.group(2)}")
                enlaces.append((url, fecha_int))
    
    # Ordenar por fecha descendente y tomar los 5 primeros
    enlaces.sort(key=lambda x: x[1], reverse=True)
    return enlaces[:5]

def procesar_excel(ruta_excel):
    """Extrae datos y asigna fecha según el nombre del archivo."""
    try:
        engine = 'xlrd' if ruta_excel.endswith('.xls') else 'openpyxl'
        df = pd.read_excel(ruta_excel, engine=engine, header=4)
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')].dropna(how='all')
        
        # Lógica de fecha basada en el nombre del archivo (8 dígitos)
        digitos = "".join(filter(str.isdigit, os.path.basename(ruta_excel)))
        if len(digitos) == 8:
            # Intentar ISO (YYYYMMDD) y luego formato español (DDMMYYYY)
            fecha_obj = pd.to_datetime(digitos, format='%Y%m%d', errors='coerce')
            if pd.isnull(fecha_obj) or fecha_obj.year < 2014:
                fecha_obj = pd.to_datetime(digitos, format='%d%m%Y', errors='coerce')
            
            if pd.notnull(fecha_obj):
                df['fecha'] = fecha_obj.strftime('%Y-%m-%d')
                return df
    except Exception:
        return None
    return None

def ejecutar_descarga_fija():
    inicializar_entorno()
    session = requests.Session()
    session.headers.update(CONFIG["HEADERS"])
    
    print("🔍 Buscando los 5 archivos más recientes...")
    objetivos = listar_ultimos_5_rars(session)
    
    if not objetivos:
        print("❌ No se encontraron archivos .rar.")
        return

    df_acumulado = []
    ext_dir = os.path.join(CONFIG["TEMP_DIR"], "extract")

    for url, _ in objetivos:
        nombre_rar = url.split('/')[-1]
        ruta_rar = os.path.join(CONFIG["TEMP_DIR"], nombre_rar)
        
        try:
            print(f"⬇️ Descargando: {nombre_rar}")
            resp = session.get(url)
            with open(ruta_rar, 'wb') as f:
                f.write(resp.content)
            
            os.makedirs(ext_dir, exist_ok=True)
            # Descomprimir usando unrar (asegúrate de tenerlo instalado en el sistema)
            subprocess.run(['unrar', 'x', '-o+', ruta_rar, ext_dir], capture_output=True, check=True)
            
            for root, _, files in os.walk(ext_dir):
                for f in files:
                    if f.lower().endswith(('.xls', '.xlsx')):
                        df_res = procesar_excel(os.path.join(root, f))
                        if df_res is not None:
                            df_acumulado.append(df_res)
            
            # Limpiar carpeta de extracción para el siguiente RAR
            shutil.rmtree(ext_dir)
            
        except Exception as e:
            print(f"⚠️ Error procesando {nombre_rar}: {e}")

    df_final = pd.DataFrame()

    if df_acumulado:
        print("📊 Uniendo datos y generando CSV...")
        df_final = pd.concat(df_acumulado, ignore_index=True)
        df_final.drop_duplicates(subset=['fecha', 'Estación'], keep='last', inplace=True)
        
        df_final.to_csv(CONFIG["ARCHIVO_SALIDA"], index=False, encoding='utf-8-sig')
        print(f"✅ Proceso finalizado. Archivo creado: {CONFIG['ARCHIVO_SALIDA']}")
        print(f"📈 Total registros: {len(df_final)}")
    else:
        print("ℹ️ No se pudo extraer ningún dato válido.")

    shutil.rmtree(CONFIG["TEMP_DIR"])

if __name__ == "__main__":
    ejecutar_descarga_fija()

🔍 Buscando los 5 archivos más recientes...
⬇️ Descargando: Aemet2026-03.rar
⬇️ Descargando: Aemet2026-02.rar
⬇️ Descargando: Aemet2026-01.rar
⬇️ Descargando: Aemet2025-12.rar
⬇️ Descargando: Aemet2025-11.rar
📊 Uniendo datos y generando CSV...
✅ Proceso finalizado. Archivo creado: /home/jovyan/work/ProyectoFinal/datos_ultimos_5_meses.csv
📈 Total registros: 124710


In [15]:
df = pd.read_csv("datos_ultimos_5_meses.csv")
df.head()

,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),fecha
0,Alforja,Tarragona,14.7 (16:30),5.2 (20:20),9.9,19 (15:40),10 (16:20),0.0,0.0,0.0,0.0,0.0,2026-03-01
1,Reus Aeropuerto,Tarragona,17.3 (13:30),8.7 (23:50),13.0,26 (15:40),17 (15:00),0.0,0.0,0.0,0.0,0.0,2026-03-01
2,Valls,Tarragona,16.2 (14:40),6.7 (23:50),11.5,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-03-01
3,Tarragona,Tarragona,16.1 (13:30),10.2 (07:00),13.2,18 (15:10),9 (14:20),0.2,0.2,0.0,0.0,0.0,2026-03-01
4,Pontons,Barcelona,12.9 (15:30),4.6 (23:59),8.7,30 (15:20),21 (14:00),0.0,0.0,0.0,0.0,0.0,2026-03-01


# API Energia electrica 

In [8]:
import requests
import pandas as pd

def obtener_balance_renovable(fecha_inicio, fecha_fin):
    url = "https://apidatos.ree.es/es/datos/balance/balance-electrico"
    
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json',
        'Host': 'apidatos.ree.es'
    }
    
    params = {
        'start_date': f'{fecha_inicio}T00:00',
        'end_date': f'{fecha_fin}T23:59',
        'time_trunc': 'day'
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        lista_dfs = []
        
        for item in data.get('included', []):
            # Filtramos estrictamente por el bloque de "Renovable"
            if item.get('type') == "Renovable":
                
                # Recorremos cada tipo de energía (Eólica, Hidráulica, etc.)
                for sub_item in item.get('attributes', {}).get('content', []):
                    tipo_nombre = sub_item.get('type')

                    # Nos quedamos solo con un tipo de energia.
                    if tipo_nombre == "Eólica":

                        valores = sub_item.get('attributes', {}).get('values', [])
                        
                        if valores:
                            temp_df = pd.DataFrame(valores)
                            temp_df['tipo_energia'] = tipo_nombre
                            lista_dfs.append(temp_df)
        
        if not lista_dfs:
            print("No se encontraron datos renovables.")
            return None
            
       # Combinamos todos los datos
        df_final = pd.concat(lista_dfs, ignore_index=True)
        
        # --- CAMBIO AQUÍ: Conversión robusta ---
        # 1. Convertimos a datetime asegurando que detecte el formato ISO de la API
        df_final['datetime'] = pd.to_datetime(df_final['datetime'], utc=True)
        
        # 2. Ahora que es "datetimelike", quitamos la zona horaria (hacemos el 'naive')
        df_final['datetime'] = df_final['datetime'].dt.tz_localize(None)
        # ---------------------------------------
        
        # Seleccionamos las columnas solicitadas
        # Asegúrate de que 'value' esté en la lista si lo añadiste manualmente
        columnas_disponibles = ['tipo_energia', 'datetime', 'percentage', 'value']
            
        df_final = df_final[columnas_disponibles]
        
        # Renombramos columnas
        df_final.columns = ['Tipo Energía', 'Fecha', 'Porcentaje (%)', 'Valor']
        
        # Multiplicamos por 100 para tener formato porcentaje (0.31 -> 31.0)
        df_final['Porcentaje (%)'] = (df_final['Porcentaje (%)'] * 100).round(2)
        
        return df_final
    else:
        print(f"Error en la API: {response.status_code}")
        return None

# --- EJECUCIÓN ---
# Nota: La API de REE a veces tiene límites de rango, un mes suele estar bien.
inicio = "2025-06-01"
fin = date.today().strftime("%Y-%m-%d")

df_energia = obtener_balance_renovable(inicio, fin)

if df_energia is not None:
    # Ordenamos cronológicamente para ver la evolución
    df_energia = df_energia.sort_values(by=['Fecha', 'Porcentaje (%)'], ascending=[True, False])
    print(df_energia.to_string(index=False))

Tipo Energía               Fecha  Porcentaje (%)      Valor
      Eólica 2025-05-31 22:00:00           31.54 116690.974
      Eólica 2025-06-01 22:00:00           39.66 187709.190
      Eólica 2025-06-02 22:00:00           29.12 111730.092
      Eólica 2025-06-03 22:00:00           23.15  95355.814
      Eólica 2025-06-04 22:00:00           22.62  96003.049
      Eólica 2025-06-05 22:00:00           17.79  76239.079
      Eólica 2025-06-06 22:00:00           20.13  75579.106
      Eólica 2025-06-07 22:00:00           25.81  90999.640
      Eólica 2025-06-08 22:00:00           24.95  99964.471
      Eólica 2025-06-09 22:00:00           27.54 108967.089
      Eólica 2025-06-10 22:00:00           37.09 161224.896
      Eólica 2025-06-11 22:00:00           22.37  95937.103
      Eólica 2025-06-12 22:00:00           24.38 106119.227
      Eólica 2025-06-13 22:00:00           21.76  80343.686
      Eólica 2025-06-14 22:00:00           38.90 163578.426
      Eólica 2025-06-15 22:00:00        

# Datos fechas

In [37]:
ultimoAño = df_clima["fecha"].tail(1).item()
ultimoAño = pd.to_numeric(str(ultimoAño).split("-")[0])

# 1. Configurar el rango de años
years = list(range(2014, (ultimoAño+1)))

# 2. Seleccionar el país (España)
es_holidays = holidays.ES(years=years)

# 3. Crear una lista de todas las fechas en ese rango
start_date = date(2014, 1, 1)
end_date = date(ultimoAño, 12, 31)
all_days = pd.date_range(start=start_date, end=end_date)

# 4. Construir el DataFrame
df_fechas = pd.DataFrame(all_days, columns=['fecha'])

# 5. Identificar festivos y nombres
df_fechas['es_festivo'] = df_fechas['fecha'].apply(lambda x: x in es_holidays)
df_fechas['fecha'] = pd.to_datetime(df_fechas['fecha']).dt.date

# 6. Nueva Columna: Tipo de Día
def clasificar_dia(row):
    if row['es_festivo']:
        return 'Festivo'
    
    if row['fecha'].weekday() >= 5:
        return 'Fin de semana'
    
    return 'Laboral'

df_fechas['tipo_dia'] = df_fechas.apply(clasificar_dia, axis=1)
df_fechas = df_fechas.drop("es_festivo", axis=1)

# 7. Asegurar que la carpeta existe y guardar
os.makedirs('data/calendario', exist_ok=True)
df_fechas.to_csv('data/calendario/calendario_festivos_2014_2026.csv', index=False, encoding='utf-8-sig')

print("¡Archivo generado con éxito!")
# Mostrar ejemplo con diferentes tipos de días
df_fechas.tail(20)

¡Archivo generado con éxito!


,fecha,tipo_dia
4728,2026-12-12,Fin de semana
4729,2026-12-13,Fin de semana
4730,2026-12-14,Laboral
4731,2026-12-15,Laboral
4732,2026-12-16,Laboral
4733,2026-12-17,Laboral
4734,2026-12-18,Laboral
4735,2026-12-19,Fin de semana
4736,2026-12-20,Fin de semana
4737,2026-12-21,Laboral


# Unir datos

In [38]:
# Unir df_extra a df_grande manteniendo la estructura de df_grande
df_clima['fecha'] = df_clima['fecha'].astype(str).str.strip()
df_fechas['fecha'] = df_fechas['fecha'].astype(str).str.strip()

df_final = pd.merge(df_clima, df_fechas, on='fecha', how='left')
df_final['fecha'] = pd.to_datetime(df_final["fecha"]).dt.date
df_final.head(50)

df_final.to_csv("Datos_final.csv", index=False, encoding='utf-8-sig')